In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Parámetros editables
N = 25
CHI = 8
IDX = 6
CHIS = [8]

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / '0' / f'mps_N{N}'
HIST_DIR = DATA_DIR / 'hist'
RESULTS_FILE = DATA_DIR / f'results_mps_N{N}_vs_T.json'

with RESULTS_FILE.open() as f:
    results = json.load(f)

T_raw = np.asarray(results['T'], dtype=float)
chi_data = {}

for chi in CHIS:
    chi_results = results['by_chi'].get(str(chi))
    if chi_results is None:
        print(f'chi={chi}: no aparece en los resultados')
        continue

    keys = ('energy', 'entropy', 'free_energy')
    values = {key: np.asarray(chi_results[key], dtype=float) for key in keys}
    n = min(len(T_raw), *(len(values[key]) for key in keys))
    if any(len(values[key]) != len(T_raw) for key in keys):
        print(f'chi={chi}: barrido parcial ({n}/{len(T_raw)} temperaturas)')

    # Cada array corresponde al prefijo del barrido original de temperaturas.
    prefix_order = np.argsort(T_raw[:n])
    T_chi = T_raw[:n][prefix_order]
    E, S2, F = (values[key][:n][prefix_order] for key in keys)
    valid = np.isfinite(T_chi) & np.isfinite(E) & np.isfinite(S2) & np.isfinite(F)
    chi_data[chi] = {
        'T': T_chi[valid], 'E': E[valid], 'S2': S2[valid], 'F': F[valid],
    }
    print(f'chi={chi}: {valid.sum()} temperaturas válidas')

T_all = np.sort(T_raw[np.isfinite(T_raw)])
print(f'N={results["N"]}; T en [{T_all.min():.2f}, {T_all.max():.2f}]')

In [ ]:
hist_path = HIST_DIR / f'hist_{IDX:04d}_chi{CHI}.npz'
if not hist_path.exists():
    raise FileNotFoundError(f'No existe el historial: {hist_path}')

with np.load(hist_path) as archive:
    if 'hist' not in archive:
        raise KeyError(f'El archivo {hist_path.name} no contiene la clave hist')
    history = np.asarray(archive['hist'], dtype=float)

if history.ndim != 2 or history.shape[1] != 3:
    raise ValueError(f'Forma inesperada en {hist_path.name}: {history.shape}; se esperaba (iteraciones, 3)')

energy_history, entropy_history, free_energy_history = history.T
steps = np.arange(len(history))
temperature = T_raw[IDX] if IDX < len(T_raw) else float('nan')
print(f'Historial: idx={IDX}, chi={CHI}, T={temperature:g}, iteraciones={len(steps)}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
series = [
    (energy_history, 'Energía'),
    (entropy_history, 'Entropía Rényi-2'),
    (free_energy_history, 'Energía libre'),
]
for ax, (values, title) in zip(axes, series):
    ax.plot(steps, values, lw=0.8)
    ax.set_xlabel('Iteración')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Valor total')
fig.suptitle(f'MPS: N={N}, chi={CHI}, idx={IDX}, T={temperature:g}')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, free_energy_history, lw=1.0, color='tab:blue')
ax.set_xlabel('Iteración')
ax.set_ylabel('Energía libre')
ax.set_title(f'Convergencia de F: N={N}, chi={CHI}, T={temperature:g}')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.optimize import brentq

J_ZZ = -1.0
h_x = -1.5
J, h = abs(J_ZZ), abs(h_x)

def exact_dos_tfim(N, J, h, dE=0.125):
    eps_fn = lambda k: 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k))
    sectors = [
        (2 * np.pi * (np.arange(N) + 0.5) / N, 0),
        (2 * np.pi * np.arange(N) / N, 1),
    ]
    energies, densities = [], []
    for k, parity in sectors:
        eps = eps_fn(k)
        n_bins = int(eps.sum() / dE) + 3
        density = np.zeros((2, n_bins))
        density[0, 0] = 1.0
        for value in eps:
            scaled = value / dE
            lo = int(scaled)
            frac = scaled - lo
            updated = density.copy()
            for p in (0, 1):
                updated[1 - p, lo:] += (1 - frac) * density[p, :n_bins - lo]
                updated[1 - p, lo + 1:] += frac * density[p, :n_bins - lo - 1]
            density = updated
        energies.append(-0.5 * eps.sum() + dE * np.arange(n_bins))
        densities.append(density[parity])

    energy_flat = np.concatenate(energies)
    density_flat = np.concatenate(densities)
    energy_grid = np.arange(energy_flat.min(), energy_flat.max() + 1.5 * dE, dE)
    density_grid, _ = np.histogram(
        energy_flat,
        bins=np.append(energy_grid - dE / 2, energy_grid[-1] + dE / 2),
        weights=density_flat,
    )
    energy_site = energy_grid / N
    dE_site = dE / N
    density_grid = density_grid / (density_grid.sum() * dE_site)
    return energy_site, density_grid, dE_site

def mre_thermo_exact(T, N, energy_dos, density_dos, dE_dos):
    if T < 1e-6:
        ground_energy = N * energy_dos.min()
        return ground_energy, 0.0, ground_energy

    def mean_energy_site(c):
        weights = np.maximum(c - energy_dos, 0.0)
        z = np.sum(density_dos * weights) * dE_dos
        if z < 1e-300:
            return np.nan
        return np.sum(density_dos * weights * energy_dos) * dE_dos / z

    def self_consistency(c):
        return c - mean_energy_site(c) - 2 * T / N

    try:
        c = brentq(self_consistency, energy_dos.min() + 1e-6,
                   energy_dos.max() + 20 * T / N + 10, xtol=1e-10)
    except ValueError:
        return np.nan, np.nan, np.nan

    mean_energy = mean_energy_site(c)
    weights = np.maximum(c - energy_dos, 0.0)
    z = np.sum(density_dos * weights) * dE_dos
    numerator = np.sum(density_dos * weights**2) * dE_dos
    if numerator <= 0 or z <= 0:
        return np.nan, np.nan, np.nan
    log_purity = np.log(numerator) - N * np.log(2) - 2 * np.log(z)
    entropy = -log_purity
    energy = N * mean_energy
    return energy, entropy, energy - T * entropy

def exact_tfim_thermo(N, J, h, beta):
    eps_fn = lambda k: 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k))
    k_ns = 2 * np.pi * (np.arange(N) + 0.5) / N
    k_r = 2 * np.pi * np.arange(N) / N
    eps_ns, eps_r = eps_fn(k_ns), eps_fn(k_r)
    log_z_ns = np.sum(np.log(2 * np.cosh(beta * eps_ns / 2)))
    log_z_r = np.sum(np.log(2 * np.cosh(beta * eps_r / 2)))
    log_z_sum = np.logaddexp(log_z_ns, log_z_r)
    log_z = log_z_sum - np.log(2)
    p_ns = np.exp(log_z_ns - log_z_sum)
    p_r = np.exp(log_z_r - log_z_sum)
    e_ns = -np.sum((eps_ns / 2) * np.tanh(beta * eps_ns / 2))
    e_r = -np.sum((eps_r / 2) * np.tanh(beta * eps_r / 2))
    energy = p_ns * e_ns + p_r * e_r
    entropy = log_z + beta * energy
    return energy / N, entropy / N

def gibbs_thermo_exact(T, N, J, h):
    if T < 0.02:
        k_ns = 2 * np.pi * (np.arange(N) + 0.5) / N
        eps_ns = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k_ns))
        energy = -0.5 * eps_ns.sum()
        return energy, 0.0, energy
    energy_site, entropy_site = exact_tfim_thermo(N, J, h, 1 / T)
    energy, entropy = N * energy_site, N * entropy_site
    return energy, entropy, energy - T * entropy

energy_dos, density_dos, dE_dos = exact_dos_tfim(N, J, h)
T_ref = np.linspace(T_all.min(), T_all.max(), 200)
E_mre, S2_mre, F_mre = np.array([
    mre_thermo_exact(T, N, energy_dos, density_dos, dE_dos) for T in T_ref
]).T
E_gib, SvN_gib, F_gib = np.array([
    gibbs_thermo_exact(T, N, J, h) for T in T_ref
]).T

colors = {2: 'tab:orange', 4: 'tab:green', 8: 'tab:purple', 16: 'tab:brown'}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
plots = [
    ('E', E_mre, E_gib, 'Energía', 'E'),
    ('S2', S2_mre, SvN_gib, 'Entropía Rényi-2', 'Entropía'),
    ('F', F_mre, F_gib, 'Energía libre', 'F'),
]
for ax, (key, mre_ref, gibbs_ref, title, ylabel) in zip(axes.flat[:3], plots):
    for chi, data in chi_data.items():
        label = f'chi={chi}' + (' (parcial)' if len(data['T']) < len(T_raw) else '')
        ax.plot(data['T'], data[key], 'o', ms=3,
                color=colors.get(chi), label=label)
    ax.plot(T_ref, mre_ref, '-', color='tab:red', lw=2, label='MRE exacto')
    ax.plot(T_ref, gibbs_ref, '--', color='tab:blue', lw=2, label='Gibbs exacto')
    ax.set_xlabel('T')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.axhline(0, color='k', ls='--', lw=1)
for chi, data in chi_data.items():
    mre_at_T = np.interp(data['T'], T_ref, F_mre)
    gap_rel = (data['F'] - mre_at_T) / np.maximum(np.abs(mre_at_T), 1e-12)
    label = f'chi={chi}' + (' (parcial)' if len(data['T']) < len(T_raw) else '')
    ax.plot(data['T'], gap_rel, 'o-', ms=3, color=colors.get(chi), label=label)
ax.set_xlabel('T')
ax.set_ylabel('(F_MPS - F_MRE) / |F_MRE|')
ax.set_title('Gap relativo respecto al MRE')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

fig.suptitle(f'N={N}', y=1.00, fontsize=13)
plt.tight_layout()
plt.show()